                 Preprocessing and feature engineering

This will involve transforming raw data into useful features that our model can learn from them effectively thus improving its predictive performance. This will involve cleaning data, removing duplicated data, encoding categorical features into numerical ones considering whether they are ordered or not.

    

In [7]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_csv(r'./BankChurners.csv')
df.head()

,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


In [ ]:
#shape of the dataset
df.shape

(10127, 23)

This dataset has 10127 rows and 23 columns

                                    Duplicate data
We will remove duplicates in our dataset since they may skew it making us to draw biased conclusions from it. Customer ID i.e ClientNUM is a unique value so useful in removing duplicated values. The shape of our dataset after removing them will proof their existence or not.

In [ ]:
df = df.loc[~df['CLIENTNUM'].index.duplicated()]
df.shape

The shape of this dataset remained the same so there aren't any duplicated values in our data. Dropping columns containing naive bayes classifiers, ClientNum since they don't have input into our analysis.

In [ ]:
#dropping columns like client number and naive_bayes_classifier from our dataset
naiveCol = [col for col in df.columns if 'Naive_Bayes'  in col] + ['CLIENTNUM']
df = df.drop(columns=naiveCol)
df.head()


,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000
3,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000


                                   Checking null values

null values in the dataset can result to biased analysis,  reduceed completeness of the dataset etc thus they require careful handling to  remove/modify them. We will check for null values in our dataset.

In [38]:
df.isnull().sum().sum()

np.int64(0)

This dataset doesn't have any null values

In [ ]:
#df['Purchase_week']  = df[['date']].applymap(lambda x: x.week if not pd.isnull(x.week) else 0)

In [40]:
df['Marital_Status'].value_counts()
df['Income_Category'].value_counts()
df['Education_Level'].value_counts()

Education_Level
Graduate         3128
High School      2013
Unknown          1519
Uneducated       1487
College          1013
Post-Graduate     516
Doctorate         451
Name: count, dtype: int64

                 Data Cleaning

Categorical columns like Education_Level, Marital_status, Income_Category has a string named 'Unknown' which might have arised due to customers privacy or data encoding errors.

Since it is quite prevalent in our data, we will replace it with a column mostrequent value(mode) since removing it will interfere with distribution of our data.

In [41]:
categorical_cols = df.select_dtypes(include='object').columns
print(categorical_cols)

Index(['Attrition_Flag', 'Gender', 'Education_Level', 'Marital_Status',
       'Income_Category', 'Card_Category'],
      dtype='object')


In [42]:
for col in categorical_cols:
    if 'Unknown' in df[col].value_counts().index:
        df[col] = df[col].replace('Unknown', df[col].mode()[0])
        

A sample using education_level to show the 'Unknown'  string has been modified to repreent the mode of education_level

In [43]:
df['Income_Category'].value_counts()

Income_Category
Less than $40K    4673
$40K - $60K       1790
$80K - $120K      1535
$60K - $80K       1402
$120K +            727
Name: count, dtype: int64

                       Label Encoding 
columns like Income_category and Education_level are ordinal data thus we will encode them taking order into consideration where low values e.g 0 will represent lowest income or uneducated while larger values like 4 will repsent highest level of education or income_category.

Attrition_Flag is a binary column so 'existing customers' will be encoded as 0 while 'Attrited customers will be encoded as 1

<bold>NB:</bold> for income category, the highest level is at 4 repsenting a customer earning more than $120K.



In [44]:
df['Attrition_Flag'] = df['Attrition_Flag'].map({'Existing Customer':0, 'Attrited Customer': 1})
income_mapping= {'Less than $40K': 0, '$40K - $60K': 1, '$60K - $80K':2, '$80K - $120K':3, '$120K +': 4}
df['Income_Category'] = df['Income_Category'].map(income_mapping)
education_level_mapping = {'Uneducated':0, 'High School' : 1, 'College': 2, 'Graduate':3, 'Post-Graduate':4, 'Doctorate':5}
df['Education_Level'] = df['Education_Level'].map(education_level_mapping)

                      One-hot encoding

Gender, Marital_status and Card_cateory are nominal data i.e order doesn't matter thus we will hot-encode them.

In [45]:
df = pd.get_dummies(df, columns=['Gender', 'Marital_Status', 'Card_Category'], drop_first=True)
df.head()

,Attrition_Flag,Customer_Age,Dependent_count,Education_Level,Income_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,...,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Gender_M,Marital_Status_Married,Marital_Status_Single,Card_Category_Gold,Card_Category_Platinum,Card_Category_Silver
0,0,45,3,1,2,39,5,1,3,12691.0,...,1144,42,1.625,0.061,True,True,False,False,False,False
1,0,49,5,3,0,44,6,1,2,8256.0,...,1291,33,3.714,0.105,False,False,True,False,False,False
2,0,51,3,3,3,36,4,1,0,3418.0,...,1887,20,2.333,0.000,True,True,False,False,False,False
3,0,40,4,1,0,34,3,4,1,3313.0,...,1171,20,2.333,0.760,False,True,False,False,False,False
4,0,40,3,0,2,21,5,1,0,4716.0,...,816,28,2.500,0.000,True,True,False,False,False,False


Finally we will save all changes to this dataset as a preprocesses csv

In [46]:
df.to_csv('./prprocessed/Banc_ChurnersProcessed.csv')